## Data Loading - International Football Results 1872 to 2026

Loading in data using kaggle. Data used: https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017/data

This dataset includes 49,393 results of international football matches starting from the very first official match in 1872 up to 2025. The matches range from FIFA World Cup to FIFI Wild Cup to regular friendly matches. The matches are strictly men's full internationals and the data does not include Olympic Games or matches where at least one of the teams was the nation's B-team, U-23 or a league select team.

**Goal of this notebook:** get from raw, messy match data to a clean
match-level table with a reliable win/draw/loss outcome per game — the
foundation that the Elo rating system (`elo_creation.ipynb`) and, later, the
Random Forest model are built on. The work here is exploratory; once an
approach is validated it gets promoted into
[`src/ml_world_cup_predictor/data_loading.py`](../src/ml_world_cup_predictor/data_loading.py).

In [ ]:
import shutil
import kagglehub
import pandas as pd
import numpy as np
import seaborn as sns
import datetime as dt
from pathlib import Path

DATA_DIRECTORY = Path("../data/international-football-results")

def get_data_path(data_directory:Path) -> Path:

    if data_directory.exists():
        path = data_directory
        print("Using cached dataset at:", path)
    else:
        download_path = kagglehub.dataset_download("martj42/international-football-results-from-1872-to-2017")
        data_directory.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(download_path, data_directory)
        path = data_directory
        print("Downloaded dataset to:", path)

    return path


def load_data(data_directory: Path) -> dict[str,pd.DataFrame]:
    frames = {file_path.stem : pd.read_csv(file_path) for file_path in sorted(data_directory.glob('*csv'))}

    for frame in frames.values():
        if 'date' in frame.columns:
            frame['date'] = pd.to_datetime(frame['date'],errors = 'coerce')
    
    return frames



frames = load_data(DATA_DIRECTORY)

results = frames['results']
shootouts = frames['shootouts']
goalscorers = frames['goalscorers']
former_names = frames['former_names']



`get_data_path`/`load_data` here are the scratch-pad version of this logic — they're later promoted into [`src/ml_world_cup_predictor/data_loading.py`](../src/ml_world_cup_predictor/data_loading.py) once validated, and reused from there in `elo_creation.ipynb`. The dataset is split across four CSVs (`results`, `shootouts`, `goalscorers`, `former_names`), downloaded once via `kagglehub` and cached locally so re-running the notebook doesn't re-download ~49k rows each time.

## Exploratory data review

Before doing any feature engineering, I wanted a consistent profile of every
column in each file: null rate, cardinality, summary statistics, and date
ranges. `dataset_review` runs this check generically across `results`,
`shootouts`, and `goalscorers` so the same exploration logic doesn't need to
be rewritten per file.

In [168]:
def string_column(df,col,cardinality_threshold):
    unique_values = df[col].nunique()
    print(f'Total Unique Values in {col} = {unique_values}')
    if unique_values <= cardinality_threshold:
        print(df[col].value_counts())
    else:
        print('\n\nTop 10 Most Frequent Values')
        print(df[col].value_counts().head(10))
        print('\n\nBottom 10 Most Frequent Values')
        print(df[col].value_counts().tail(20))

def number_column(df,col):
    print(f'\n\n=== Summary Statistic {col}===')
    print(f' Max = {df[col].max():.2f}')
    print(f' Mean = {df[col].mean():.2f}')
    #print(f' Mode = {df[col].mode():.2f}')
    print(f' Median = {df[col].median():.2f}')
    print(f' Min = {df[col].min():.2f}')

def date_column(df,col):

    df = df.copy()

    print(f'\n\n=== Summary Statistic {col}===')

    print(f'Earliest Date == {df[col].min()}')
    print(f'Latest Date == {df[col].max()}')

    df['month'] = df[col].dt.month
    df['year'] = df[col].dt.year
    

    print(df[col].value_counts().head(25))

    print('\n\n',df['year'].value_counts().head(25))

    print('\n\n',df['month'].value_counts())

    #print(cross_count.to_string)

def column_review(df,col,cardinality_threshold = 100):

    nulls = (df[col].isna()).sum()
    pct_null = nulls/(len(df[col]))
    print(f'{col} has {pct_null:.1%} nulls ({nulls} out of {len(df[col]):,})')

    if pd.api.types.is_string_dtype(df[col]):
        string_column(df,col,cardinality_threshold)
    
    if pd.api.types.is_any_real_numeric_dtype(df[col]) or pd.api.types.is_float_dtype(df[col]):
        number_column(df,col)

    if pd.api.types.is_datetime64_any_dtype(df[col]):
        date_column(df,col)
        
            


def dataset_review(df,cardinality_threshold = 100):

    
    print(f'Totals Rows = {len(df)}')
    print(f'Totals Columns = {len(df.columns)}') 
    print(f'Columns = {list(df.columns)}')
    print(df.dtypes)

    print('\n\n === Example Row ===')
    print(df.iloc[-1])

    for column in df.columns:

        column_review(df,column)



In [169]:
dataset_review(results)

Totals Rows = 49477
Totals Columns = 9
Columns = ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']
date          datetime64[us]
home_team                str
away_team                str
home_score           float64
away_score           float64
tournament               str
city                     str
country                  str
neutral                 bool
dtype: object


 === Example Row ===
date          2026-06-27 00:00:00
home_team                 Croatia
away_team                   Ghana
home_score                    NaN
away_score                    NaN
tournament         FIFA World Cup
city                 Philadelphia
country             United States
neutral                      True
Name: 49476, dtype: object
date has 0.0% nulls (0 out of 49,477)


=== Summary Statistic date===
Earliest Date == 1872-11-30 00:00:00
Latest Date == 2026-06-27 00:00:00
date
2012-02-29    66
2016-03-29    64
2025-06-10    62
2025-03-25    6

In [170]:
dataset_review(shootouts)

Totals Rows = 678
Totals Columns = 5
Columns = ['date', 'home_team', 'away_team', 'winner', 'first_shooter']
date             datetime64[us]
home_team                   str
away_team                   str
winner                      str
first_shooter               str
dtype: object


 === Example Row ===
date             2026-06-06 00:00:00
home_team                  Lithuania
away_team                     Latvia
winner                     Lithuania
first_shooter              Lithuania
Name: 677, dtype: object
date has 0.0% nulls (0 out of 678)


=== Summary Statistic date===
Earliest Date == 1967-08-22 00:00:00
Latest Date == 2026-06-06 00:00:00
date
2016-06-03    5
2024-03-26    5
1986-10-01    4
2018-06-09    4
2021-07-06    4
2026-03-30    4
2018-06-03    3
2020-10-08    3
1973-06-14    2
1973-07-26    2
1984-03-14    2
1984-12-16    2
1985-12-25    2
1986-06-21    2
1989-04-23    2
1990-12-17    2
1991-01-21    2
1995-07-17    2
1995-12-02    2
1996-06-22    2
1996-06-26    2
1996

In [171]:
dataset_review(goalscorers)

Totals Rows = 47690
Totals Columns = 8
Columns = ['date', 'home_team', 'away_team', 'team', 'scorer', 'minute', 'own_goal', 'penalty']
date         datetime64[us]
home_team               str
away_team               str
team                    str
scorer                  str
minute              float64
own_goal               bool
penalty                bool
dtype: object


 === Example Row ===
date            2026-06-18 00:00:00
home_team               Switzerland
away_team    Bosnia and Herzegovina
team         Bosnia and Herzegovina
scorer                 Ermin Mahmić
minute                         90.0
own_goal                      False
penalty                       False
Name: 47689, dtype: object
date has 0.0% nulls (0 out of 47,690)


=== Summary Statistic date===
Earliest Date == 1916-07-02 00:00:00
Latest Date == 2026-06-18 00:00:00
date
2011-10-11    145
2008-10-11    136
2008-09-06    135
2023-11-16    135
2004-09-08    132
2004-10-13    128
2011-09-02    125
2011-09-06    12

**Findings from the review:**
- `results` has 49,477 matches, with `home_score`/`away_score` null for 44
  rows — these are upcoming/unplayed fixtures (including the entire 2026
  World Cup draw), not data quality issues, and need to be separated from
  played matches before building any features.
- `tournament` has 200 distinct values, dominated by `Friendly` (18k+) and
  World Cup/continental qualifiers — this long tail matters later, since a
  friendly and a World Cup final shouldn't move a team's rating by the same
  amount.
- `shootouts` records the penalty-shootout winner separately from `results`,
  so a drawn match that was actually settled on penalties needs to be
  reconciled against this file to get the real W/L outcome.
- Team names aren't stable over time (e.g. historical entities, country
  renames) — `former_names` is needed to map historical squad names onto
  their current name before aggregating by team.

In [255]:
len(results[results['home_score'].isna()])

missing_match = {
    'date':dt.datetime(2024,10,27),
    'home_team':'Sierra Leone',
    'away_team':'Liberia',
    'home_score':1,
    'away_score':2,
    'tournament':'African Cup of Nations qualification',
    'city':'Monrovia',
    'country':'Liberia',
    'netural': False
}

row = pd.DataFrame(missing_match,index = [0])

results_new = pd.concat([results,row],ignore_index=True)

results_new = results_new.sort_values(by = 'date').reset_index()

results_new

,index,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,netural
0,0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,NaN
1,1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,NaN
2,2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,NaN
3,3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,NaN
4,4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...
49473,49472,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True,NaN
49474,49473,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True,NaN
49475,49474,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True,NaN
49476,49475,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True,NaN


**Data gap found:** a Sierra Leone vs Liberia African Cup of Nations
qualifier (2024-10-27) was missing from the source dataset entirely, despite
having been played, while genuinely upcoming/unplayed fixtures correctly
show `NaN` scores. Found by cross-checking results showing null scores
against fixture dates already in the past. The missing row is manually
appended here rather than dropped, so it isn't silently lost from any
team's match history or rating.

## Reconciling shootout results

`results` records penalty-shootout matches as draws on the scoreboard
(e.g. 1-1), with the actual winner only recoverable from the separate
`shootouts` file. To get a clean W/L/D outcome per match, the two files are
joined on `date` + `home_team` + `away_team`, and the shootout winner is
attached to the corresponding match. One shootout (Saare County vs Åland
Islands, 2011) doesn't have a matching row in `results` and is left
unmatched rather than guessed at.

In [ ]:


not_played = results_new[results_new['home_score'].isna()]
played = results_new[~results_new['home_score'].isna()]

played = played.assign(shootout_id = played[['date','home_team','away_team']].apply(
    lambda row: '_'.join([str(each) for each in row]), axis = 1
))

shootouts = shootouts.assign(shootout_id = shootouts[['date','home_team','away_team']].apply(
    lambda row: '_'.join([str(each) for each in row]), axis = 1
))

shootouts_ids = list(shootouts['shootout_id'].unique())

shootout_dict = pd.Series(shootouts['winner'].values, index = shootouts['shootout_id']).to_dict()

matches_with_shootouts = played[played['shootout_id'].isin(shootouts_ids)]

matches_with_shootouts = list(matches_with_shootouts['shootout_id'].unique())

print(len(shootouts_ids),len(matches_with_shootouts))

print(set(shootouts_ids)-set(matches_with_shootouts))


played['shootout_winner'] = played['shootout_id'].map(shootout_dict)


678 677
{'2011-06-29 00:00:00_Saare County_Åland Islands'}


In [257]:
played['goal_diff'] = np.abs(played['home_score']-played['away_score'])
played['year'] = played['date'].dt.year
played_long = played.melt(
    id_vars=['date','year','tournament'],
    value_vars=['home_team','away_team'],
    value_name='team'
)


wc_matches = played_long[(played_long['tournament'] == 'FIFA World Cup') & (played_long['year'] == 2026)]
wc_teams = wc_matches['team'].unique()
wc_former_names = former_names[former_names['current'].isin(wc_teams)]

name_map = dict(zip(wc_former_names['former'], wc_former_names['current']))

played['home_team'] = played['home_team'].replace(name_map)
played['away_team'] = played['away_team'].replace(name_map)

match_conditions = [
    played['home_score'] > played['away_score'],
    played['home_score'] < played['away_score'],
    played['shootout_winner'] == played['home_team'],
    played['shootout_winner'] == played['away_team']
]  

choices = ['W','L','W','L']
played['result'] = np.select(match_conditions,choices,default='D')




## Deriving match outcome and normalising team names

With shootout winners attached, each match gets a single `result` column
(`W`/`L`/`D` from the home team's perspective) and a `goal_diff` column,
which becomes the basis for outcome labels and Elo updates later. Team
names for the 2026 World Cup squads are also normalised against
`former_names`, so a team that competed under a former name in older
matches is still recognised as the same team when building its rating
history.

## Exploring "lineal champions"

As a sanity check on the cleaned `result` data (and a fun aside before the
core Elo work), `lineal_champions` tracks a boxing-style "champion" who
holds the title until beaten by a challenger they actually played. This is
purely exploratory — it isn't used as a model feature — but it's a good way
to spot-check that match results are being interpreted correctly across the
full history.

In [243]:

def lineal_champions(df):
    current_champion = None
    reign_log = []

    for row in df.itertuples(index = True,name='Match'):
        
        if current_champion is not None and current_champion not in (row.home_team, row.away_team):
            continue
        
        if row.result == 'W':
            challenger = row.home_team
        elif row.result == 'L':
            challenger = row.away_team
        else:
            continue

        if challenger != current_champion:
            reign_log.append({
                'date':row.date,
                'new_champion':challenger,
                'previous_champion':current_champion
            })
            current_champion = challenger

    reigns = pd.DataFrame(reign_log)
    reigns['reign_start'] = pd.to_datetime(reigns['date'])
    reigns['reign_end'] = reigns['reign_start'].shift(-1)
    reigns['reign_end'] = reigns['reign_end'].fillna(df['date'].max())
    reigns['days_reign'] = (reigns['reign_end'] - reigns['reign_start']).dt.days

    return reigns


In [ ]:


current_champion = None
reign_log = []

for row in played.itertuples(index = True,name='Match'):
    
    if current_champion is not None and current_champion not in (row.home_team, row.away_team):
        continue
    
    if row.result == 'W':
        challenger = row.home_team
    elif row.result == 'L':
        challenger = row.away_team
    else:
        continue

    if challenger != current_champion:
        reign_log.append({
            'date':row.date,
            'new_champion':challenger,
            'previous_champion':current_champion
        })
        current_champion = challenger

reigns = pd.DataFrame(reign_log)
reigns['reign_start'] = pd.to_datetime(reigns['date'])
reigns['reign_end'] = reigns['reign_start'].shift(-1)
reigns['reign_end'] = reigns['reign_end'].fillna(played['date'].max())
reigns['days_reign'] = (reigns['reign_end'] - reigns['reign_start']).dt.days

reigns


**Data quality issue found:** running the lineal champions check surfaced a
duplicate Zimbabwe vs Malawi result (row 29509) that doesn't match the
official record between the two sides — the duplicate is dropped before
re-running the check. This is a good example of why a derived sanity check
like this is worth keeping in the notebook even though it isn't a model
input: it catches row-level errors that column-level profiling (nulls,
ranges, cardinality) doesn't.

In [245]:
#def build_elo_trend_by_team(df,team,base = 1000,runway = 30):

zim_mask = (played['home_team'] == 'Zimbabwe') | ((played['away_team'] == 'Zimbabwe'))
mal_mask = (played['home_team'] == 'Malawi') | ((played['away_team'] == 'Malawi'))


def two_team_filter(df,team_a,team_b):

    team_a_mask = (df['home_team'] == team_a) | ((df['away_team'] == team_a))
    team_b_mask = (played['home_team'] == team_b) | ((played['away_team'] == team_b))

    df_new = df[team_a_mask & team_b_mask]

    return df_new

played[zim_mask & mal_mask]


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,shootout_id,shootout_winner,goal_diff,year,result
7215,1967-11-12,Zimbabwe,Malawi,1.0,0.0,Friendly,Salisbury,Southern Rhodesia,False,1967-11-12 00:00:00_Zimbabwe_Malawi,NaN,1.0,1967,W
7601,1969-02-01,Malawi,Zimbabwe,2.0,2.0,Friendly,Zomba,Malawi,False,1969-02-01 00:00:00_Malawi_Zimbabwe,NaN,0.0,1969,D
7608,1969-02-08,Zimbabwe,Malawi,4.0,0.0,Friendly,Salisbury,Southern Rhodesia,False,1969-02-08 00:00:00_Zimbabwe_Malawi,NaN,4.0,1969,W
7611,1969-02-09,Zimbabwe,Malawi,1.0,1.0,Friendly,Salisbury,Southern Rhodesia,False,1969-02-09 00:00:00_Zimbabwe_Malawi,NaN,0.0,1969,D
7934,1969-12-01,Malawi,Zimbabwe,2.0,2.0,Friendly,Blantyre,Malawi,False,1969-12-01 00:00:00_Malawi_Zimbabwe,NaN,0.0,1969,D
12310,1980-07-06,Malawi,Zimbabwe,0.0,1.0,Friendly,Blantyre,Malawi,False,1980-07-06 00:00:00_Malawi_Zimbabwe,NaN,1.0,1980,L
12312,1980-07-08,Malawi,Zimbabwe,1.0,1.0,Friendly,Blantyre,Malawi,False,1980-07-08 00:00:00_Malawi_Zimbabwe,NaN,0.0,1980,D
12473,1980-10-26,Malawi,Zimbabwe,0.0,1.0,African Cup of Nations qualification,Blantyre,Malawi,False,1980-10-26 00:00:00_Malawi_Zimbabwe,NaN,1.0,1980,L
12501,1980-11-09,Zimbabwe,Malawi,1.0,1.0,African Cup of Nations qualification,Salisbury,Zimbabwe,False,1980-11-09 00:00:00_Zimbabwe_Malawi,NaN,0.0,1980,D
12730,1981-04-20,Zimbabwe,Malawi,4.0,0.0,Friendly,Bulawayo,Zimbabwe,False,1981-04-20 00:00:00_Zimbabwe_Malawi,NaN,4.0,1981,W


In [ ]:
played_zimbabwe_fix = played.drop(29509)

missing_match = {
    'date':dt.datetime(2024,10,27),
    'home_team':'Sierra Leone',
    'away_team':'Liberia',
    'home_score':1,
    'away_score':2,
    'tournament':'African Cup of Nations qualification',
    'city':'Monrovia',
    'country':'Liberia',
    'netural': False
}

reign = lineal_champions(played_zimbabwe_fix)


new_champion
Scotland               12997
England                 7497
Argentina               2895
Northern Ireland        2709
France                  2343
Netherlands             2193
Russia                  2096
Wales                   1820
Uruguay                 1694
Brazil                  1581
Germany                 1420
Chile                   1418
Colombia                1348
Italy                   1334
Spain                   1216
Switzerland             1181
Sweden                  1025
Hungary                 1008
Peru                     960
Austria                  816
Croatia                  658
Paraguay                 461
Bulgaria                 422
Czechoslovakia           391
Israel                   336
Portugal                 314
Mexico                   312
Angola                   310
Greece                   308
Czech Republic           257
Mozambique               216
Bolivia                  214
Algeria                  205
Kosovo                   204
I

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
6824,1966-11-12,Sierra Leone,Liberia,1.0,1.0,Friendly,Freetown,Sierra Leone,False
6832,1966-11-19,Liberia,Sierra Leone,2.0,0.0,Friendly,Monrovia,Liberia,False
10520,1976-04-25,Sierra Leone,Liberia,1.0,0.0,Friendly,Freetown,Sierra Leone,False
13440,1982-11-07,Liberia,Sierra Leone,1.0,0.0,Friendly,Monrovia,Liberia,False
15488,1986-10-05,Sierra Leone,Liberia,2.0,1.0,African Cup of Nations qualification,Freetown,Sierra Leone,False
15511,1986-10-19,Liberia,Sierra Leone,1.0,1.0,African Cup of Nations qualification,Monrovia,Liberia,False
16435,1988-11-26,Liberia,Sierra Leone,0.0,0.0,Friendly,Monrovia,Liberia,False
20761,1995-12-17,Sierra Leone,Liberia,1.0,0.0,Friendly,Freetown,Sierra Leone,False
20785,1996-01-06,Sierra Leone,Liberia,2.0,1.0,Friendly,Freetown,Sierra Leone,False
25239,2001-02-25,Liberia,Sierra Leone,1.0,0.0,FIFA World Cup qualification,Paynesville,Liberia,False
